# 03 · Feature engineering
Lags, rolling means, holidays, prices, promotions. **Every sales-based feature is shifted by ≥ 28 days**
(the forecast horizon), so one *direct* model can predict the whole window and nothing leaks from the future.

In [1]:
import sys; sys.path.insert(0, "..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, json
from src.config import *
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3})

from src.features import add_features, feature_columns, time_split
df = pd.read_parquet(PROC_DIR / "long.parquet")
feat = add_features(df)
FEATS = feature_columns(feat)
print(len(FEATS), "features:"); print(FEATS)

33 features:
['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'wday', 'month', 'event_type_1', 'sell_price', 'snap', 'lag_28', 'lag_35', 'lag_42', 'lag_49', 'lag_56', 'dow_mean_4', 'rmean_7', 'rmean_14', 'rmean_28', 'rmean_56', 'rstd_28', 'rzero_28', 'price_rel', 'price_chg_7', 'price_vs_mean', 'promo_proxy', 'dow', 'dom', 'weekofyear', 'is_weekend', 'is_event', 'days_to_event', 'days_since_event']


## Leakage check
For one series, rebuild lag_28 by hand and confirm the feature equals sales exactly 28 days earlier.

In [2]:
s = feat[feat.id == feat.id.iloc[0]].reset_index(drop=True)
i = 100
assert s.loc[i, "lag_28"] == s.loc[i - HORIZON, "sales"], "lag_28 mismatch"
assert abs(s.loc[i, "rmean_7"] - s.loc[i - HORIZON - 6: i - HORIZON, "sales"].mean()) < 1e-6
print("Leakage check passed: features at day T only use sales up to T-28.")

Leakage check passed: features at day T only use sales up to T-28.


## Train / validation / test windows
Test = last 28 days (the M5 evaluation window). Validation = the 28 days before it (early stopping,
calibration of prediction intervals). Training = everything earlier.

In [3]:
tr, va, te = time_split(feat)
for name, part in [("train", tr), ("valid", va), ("test", te)]:
    print(f"{name:5s} {part.date.min().date()} → {part.date.max().date()}  rows={len(part):,}")
feat.to_parquet(PROC_DIR / "features.parquet")
json.dump(FEATS, open(PROC_DIR / "feature_list.json", "w"))
feat[FEATS].describe().T[["mean", "std", "min", "max"]].round(2).head(20)

train 2014-04-09 → 2016-02-11  rows=404,400
valid 2016-02-12 → 2016-03-10  rows=16,800
test  2016-03-11 → 2016-04-07  rows=16,800


,mean,std,min,max
wday,4.00,2.00,1.00,7.00
month,6.52,3.45,1.00,12.00
sell_price,3.50,1.74,0.74,12.55
snap,0.33,0.47,0.00,1.00
lag_28,1.38,2.17,0.00,82.00
lag_35,1.37,2.16,0.00,82.00
lag_42,1.37,2.16,0.00,82.00
lag_49,1.36,2.15,0.00,82.00
lag_56,1.36,2.15,0.00,82.00
dow_mean_4,1.37,1.67,0.00,42.00
